In [1]:
import pygame
import random
import math
import sys

pygame.init()

WIDTH, HEIGHT = 1100, 650
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Boat Journey With Turning")

clock = pygame.time.Clock()
font = pygame.font.SysFont("arial", 30)

# Colors
SKY_TOP = (135, 206, 250)
SKY_BOTTOM = (255, 210, 170)
SAND = (194, 178, 128)
GREEN = (34, 139, 34)
BROWN = (139, 69, 19)
DARK_BROWN = (100, 50, 20)
GRAY = (200, 200, 200)
ROPE_COLOR = (150, 120, 80)
SUN_COLOR = (255, 200, 0)
CLOUD_COLOR = (255, 255, 255)

wave_time = 0

def get_wave_y(x, t):
    return 430 + math.sin(x * 0.02 + t) * 10 + math.sin(x * 0.04 + t*1.5) * 5


# 🐟 Fish Class
class Fish:
    def __init__(self):
        self.size = random.randint(25, 55)
        self.color = random.choice([
            (255,120,50),
            (50,200,255),
            (255,50,150),
            (255,255,0),
            (100,255,100)
        ])
        self.x = random.randint(120, WIDTH-120)
        self.speed = random.uniform(1, 2)
        self.direction = random.choice([-1, 1])
        self.tail = 0

        self.jumping = False
        self.jump_time = 0

        self.base_y = get_wave_y(self.x, wave_time) + 20
        self.y = self.base_y

    def start_jump(self):
        if not self.jumping:
            self.jumping = True
            self.jump_time = 0

    def move(self, boat_rect):
        self.x += self.speed * self.direction

        if self.x > WIDTH-100 or self.x < 100:
            self.direction *= -1

        self.tail += 0.3

        if not self.jumping:
            self.base_y = get_wave_y(self.x, wave_time) + 20
            self.y = self.base_y

        if self.jumping:
            self.jump_time += 0.25
            self.y = self.base_y - abs(math.sin(self.jump_time)) * 60

            fish_rect = pygame.Rect(self.x, self.y,
                                    self.size, self.size//2)

            # Stop jump if hits boat
            if fish_rect.colliderect(boat_rect):
                self.jumping = False
                self.y = self.base_y

            if self.jump_time > math.pi:
                self.jumping = False
                self.y = self.base_y

    def draw(self):
        body = pygame.Rect(self.x, self.y,
                           self.size, self.size//2)
        pygame.draw.ellipse(screen, self.color, body)

        offset = math.sin(self.tail) * 5

        if self.direction == 1:
            points = [(self.x, self.y+self.size//4),
                      (self.x-15, self.y+self.size//4+offset),
                      (self.x, self.y+self.size//2)]
            pygame.draw.polygon(screen, self.color, points)
            pygame.draw.circle(screen, (0,0,0),
                               (self.x + self.size - 10,
                                self.y + self.size//4), 3)
        else:
            points = [(self.x+self.size, self.y+self.size//4),
                      (self.x+self.size+15, self.y+self.size//4+offset),
                      (self.x+self.size, self.y+self.size//2)]
            pygame.draw.polygon(screen, self.color, points)
            pygame.draw.circle(screen, (0,0,0),
                               (self.x + 10,
                                self.y + self.size//4), 3)


fishes = [Fish() for _ in range(10)]

boat_x = 100
boat_speed = 4
boat_direction = 1
arrived = False

running = True

while running:
    clock.tick(60)

    # 🌅 Sky Gradient
    for y in range(0, 430):
        ratio = y / 430
        r = SKY_TOP[0]*(1-ratio) + SKY_BOTTOM[0]*ratio
        g = SKY_TOP[1]*(1-ratio) + SKY_BOTTOM[1]*ratio
        b = SKY_TOP[2]*(1-ratio) + SKY_BOTTOM[2]*ratio
        pygame.draw.line(screen,
                         (int(r),int(g),int(b)),
                         (0,y),(WIDTH,y))

    # ☀ Sun
    pygame.draw.circle(screen, SUN_COLOR, (900,120), 50)

    # ☁ Clouds
    pygame.draw.circle(screen, CLOUD_COLOR, (200,120), 35)
    pygame.draw.circle(screen, CLOUD_COLOR, (240,120), 45)
    pygame.draw.circle(screen, CLOUD_COLOR, (280,120), 35)

    pygame.draw.circle(screen, CLOUD_COLOR, (500,160), 30)
    pygame.draw.circle(screen, CLOUD_COLOR, (530,160), 40)
    pygame.draw.circle(screen, CLOUD_COLOR, (560,160), 30)

    wave_time += 0.05

    # 🏖 Beaches
    pygame.draw.rect(screen, SAND, (0, 430, 80, HEIGHT))
    pygame.draw.rect(screen, GREEN, (0, 430, 80, 25))
    pygame.draw.rect(screen, SAND, (WIDTH-80, 430, 80, HEIGHT))
    pygame.draw.rect(screen, GREEN, (WIDTH-80, 430, 80, 25))

    # 🌊 Water layers
    for layer in [(445,0.6,18,(0,80,140)),
                  (438,1.0,12,(0,105,170)),
                  (430,1.8,6,(0,130,200))]:

        base, speed, height, color = layer
        points = []
        for x in range(80, WIDTH-80):
            y = base + math.sin(x*0.02 + wave_time*speed)*height
            points.append((x,y))
        points.append((WIDTH-80, HEIGHT))
        points.append((80, HEIGHT))
        pygame.draw.polygon(screen, color, points)

    # 🎮 Boat movement
    keys = pygame.key.get_pressed()
    if not arrived:
        if keys[pygame.K_RIGHT]:
            boat_x += boat_speed
        if keys[pygame.K_LEFT]:
            boat_x -= boat_speed

    boat_x = max(100, min(WIDTH-100, boat_x))
    boat_y = get_wave_y(boat_x, wave_time)
    tilt = math.sin(wave_time + boat_x*0.02) * 4

    # 🚤 Boat
    boat_surface = pygame.Surface((130,60), pygame.SRCALPHA)

    pygame.draw.polygon(boat_surface, BROWN,
        [(10,35),(25,20),(100,20),(120,35),(100,45),(25,45)])
    pygame.draw.polygon(boat_surface, DARK_BROWN,
        [(20,38),(110,38),(100,45),(25,45)])
    pygame.draw.line(boat_surface,(90,50,20),(25,20),(100,20),3)
    pygame.draw.polygon(boat_surface,GRAY,
        [(65,5),(65,20),(95,20)])

    if boat_direction == -1:
        boat_surface = pygame.transform.flip(
            boat_surface, True, False)

    rotated_boat = pygame.transform.rotate(
        boat_surface, tilt)
    boat_rect = rotated_boat.get_rect(
        center=(boat_x, boat_y - 8))
    screen.blit(rotated_boat, boat_rect)

    # 🐟 Fish System (FIXED)
    if not arrived:

        jumping_now = [f for f in fishes if f.jumping]

        if len(jumping_now) < 4:
            if random.randint(1, 60) == 1:
                not_jumping = [f for f in fishes if not f.jumping]
                if not_jumping:
                    random.choice(not_jumping).start_jump()

        for fish in fishes:
            fish.move(boat_rect)
            fish.draw()

    # 🎯 Arrival detection
    if boat_direction == 1 and boat_x >= WIDTH-120:
        arrived = True
    elif boat_direction == -1 and boat_x <= 120:
        arrived = True

    # ⚓ Dock System
    if arrived:
        pole_x = WIDTH-40 if boat_direction==1 else 40
        pygame.draw.rect(screen,(120,70,30),
                         (pole_x-5,390,10,40))

        pygame.draw.line(screen,ROPE_COLOR,
                         (boat_rect.centerx,
                          boat_rect.centery),
                         (pole_x,400),3)

        anchor_x = boat_rect.centerx - 15
        anchor_top = boat_rect.centery + 10
        anchor_bottom = 520

        pygame.draw.line(screen,GRAY,
                         (anchor_x,anchor_top),
                         (anchor_x,anchor_bottom),3)
        pygame.draw.line(screen,GRAY,
                         (anchor_x-8,anchor_bottom),
                         (anchor_x+8,anchor_bottom),4)
        pygame.draw.arc(screen,GRAY,
                        (anchor_x-12,
                         anchor_bottom-8,24,20),
                        math.pi,2*math.pi,3)

        text = font.render(
            "Boat Tied! Press C to Continue or ESC to Exit",
            True,(255,255,255))
        screen.blit(text,
            (WIDTH//2-text.get_width()//2,260))

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        if event.type == pygame.KEYDOWN:
            if arrived:
                if event.key == pygame.K_ESCAPE:
                    pygame.quit()
                    sys.exit()
                if event.key == pygame.K_c:
                    boat_direction *= -1
                    arrived = False
                    fishes = [Fish() for _ in range(10)]

    pygame.display.update()

pygame.quit()

pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


SystemExit: 

c:\Users\Lenovo\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
